<a href="https://colab.research.google.com/github/Dilukshika-Sasitharan/Statistical-Learning-e23355/blob/main/E23355_Assignment_7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ASSIGNMENT 7d : STRUCTURAL HEALTH MONITORING**
# **E23355**

---

## Task 1 — Prior Belief Boundaries

The initial prior distribution for the unknown remaining stiffness efficiency is

$$
\Theta \sim \mathrm{Beta}(8,1.5)
$$

with probability density function

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{B(8,1.5)}
\theta^{7}(1-\theta)^{0.5},
\qquad 0<\theta<1.
$$

For numerical implementation, the computational grid is restricted to

$$
0.01 \le \theta \le 1.0.
$$

This avoids the logarithmic singularity at **θ = 0** when evaluating the log-normal likelihood.

### Analytical Expected Value

The expected value of the Beta prior is

$$
E[\Theta^{(0)}]
=
\frac{\alpha_0}{\alpha_0+\beta_0}
=
\frac{8}{8+1.5}
=
\frac{8}{9.5}
\approx 0.8421.
$$

### Engineering Interpretation

A **Beta(8,1.5)** prior places most of its probability mass near **θ = 1**, indicating that the structure is expected to be in a healthy condition before any sensor measurements are collected. At the same time, it assigns a small but non-zero probability to lower stiffness values, allowing the Bayesian update process to adapt if the observed measurements indicate structural degradation.

The lower grid limit of **0.01** is used solely for numerical stability and has a negligible effect on the posterior because the prior density near **θ = 0** is extremely small.

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# ---------------------------------------------------------
# Prior Distribution: Beta(8, 1.5)
# ---------------------------------------------------------

alpha0 = 8
beta0 = 1.5

# Computational grid
theta_grid = np.linspace(0.01, 1.0, 500)

# Prior probability density
prior = stats.beta.pdf(theta_grid, alpha0, beta0)

# Analytical prior mean
E_prior = alpha0 / (alpha0 + beta0)

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior,
        mode="lines",
        name="Prior: Beta(8,1.5)"
    )
)

fig.add_vline(
    x=E_prior,
    line_dash="dash",
    annotation_text=f"E[θ] = {E_prior:.3f}",
    annotation_position="top left"
)

fig.update_layout(
    title="Initial Prior Distribution of Remaining Stiffness Efficiency",
    xaxis_title="Remaining Stiffness Efficiency (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

print(f"Prior Distribution : Beta({alpha0}, {beta0})")
print(f"Analytical Prior Mean = {E_prior:.4f}")

Prior Distribution : Beta(8, 1.5)
Analytical Prior Mean = 0.8421


## Task 2 — Structural Likelihood Formulation

The structural measurement model assumes that the measured stiffness is affected by multiplicative log-normal noise:

$$
y_k
=
\theta K_{\mathrm{nominal}} e^{\epsilon_k},
\qquad
\epsilon_k \sim N(0,\sigma^2).
$$

Taking the natural logarithm of both sides gives

$$
\ln(y_k)
\sim
N\!\left(
\ln(\theta K_{\mathrm{nominal}}),
\sigma^2
\right).
$$

Therefore, the measurement **yₖ** follows a log-normal distribution with scale parameter **θKₙₒₘᵢₙₐₗ**.

### Single-Observation Likelihood

The likelihood contribution of a single sensor measurement is

$$
L(y_k \mid \theta)
=
\frac{1}
{y_k \sigma \sqrt{2\pi}}
\exp\!\left(
-
\frac{
\left[
\ln(y_k)
-
\ln(\theta K_{\mathrm{nominal}})
\right]^2
}
{2\sigma^2}
\right).
$$

### Joint Likelihood

Assuming that the sensor measurements are conditionally independent given the unknown stiffness parameter **θ**, the joint likelihood for the observation history

$$
\mathbf{y}^{(k)}
=
(y_1, y_2, \ldots, y_k)
$$

is

$$
L(\mathbf{y}^{(k)} \mid \theta)
=
\prod_{i=1}^{k}
\frac{1}
{y_i \sigma \sqrt{2\pi}}
\exp\!\left(
-
\frac{
\left[
\ln(y_i)
-
\ln(\theta K_{\mathrm{nominal}})
\right]^2
}
{2\sigma^2}
\right).
$$

## Task 3 — Why No Closed Form Exists, and the Recursive Update

A conjugate Bayesian update (such as the Beta–Binomial model) exists only when the likelihood function and the prior belong to conjugate families. In that case, multiplying the likelihood by the prior produces a posterior distribution that belongs to the same family as the prior.

In this problem:

- The prior is a Beta distribution with kernel

$$
\theta^{\alpha-1}(1-\theta)^{\beta-1}.
$$

- The likelihood is log-normal. As a function of θ, it has the form

$$
\exp\!\left[-c\left(\ln\theta-d\right)^2\right],
$$

where **c** and **d** are constants determined by the measurement model.

Since the likelihood contains the logarithm of θ inside a squared exponential term, multiplying it by the Beta kernel does **not** produce another Beta distribution (or any other standard conjugate family). Therefore, no closed-form posterior distribution exists.

Consequently, the posterior distribution must be obtained numerically using a grid-based Bayesian update.

### Recursive Bayesian Update

The posterior distribution after observing the first **k** measurements is

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
\left(
\theta
\mid
\mathbf{y}^{(k)}
\right)
\propto
L(y_k\mid\theta)\,
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
\left(
\theta
\mid
\mathbf{y}^{(k-1)}
\right).
$$

At each iteration, the normalized posterior from step **k − 1** becomes the prior for step **k**.

The initial prior distribution is

$$
f_{\Theta\mid\mathbf{Y}^{(0)}}
=
f_{\Theta}^{(0)}
=
\mathrm{Beta}(8,1.5).
$$

## Task 4 — Running Point Estimates (Definite Integral Equations)

Since the posterior distribution does not have a closed-form analytical expression, both point estimators must be computed numerically over the bounded domain.

### Running Posterior Mean

The Bayesian estimate under squared-error loss is the expected value of the current posterior distribution:

$$
\hat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\int_{0.01}^{1.0}
\theta\,
f_{\Theta\mid\mathbf{Y}^{(k)}}
\left(
\theta
\mid
\mathbf{y}^{(k)}
\right)
\,d\theta.
$$

### Running Maximum A Posteriori (MAP) Estimate

The MAP estimate is the value of θ that maximizes the current posterior density:

$$
\hat{\theta}_{\mathrm{MAP}}^{(k)}
=
\operatorname*{arg\,max}_{\,0.01\le\theta\le1.0}
f_{\Theta\mid\mathbf{Y}^{(k)}}
\left(
\theta
\mid
\mathbf{y}^{(k)}
\right).
$$

Since the posterior distribution is non-conjugate, neither estimator has a closed-form analytical solution. Therefore, both the posterior mean and the MAP estimate are evaluated numerically using the normalized posterior distribution on the computational grid.

## Task 5 — Algorithmic Grid Approximation and Normalization

Since the posterior distribution has no closed-form analytical solution, it is approximated numerically on a fixed grid over the bounded parameter space.

### Step 1 — Grid Construction

Create a grid of **M** equally spaced values over the physical domain

$$
0.01 \le \theta \le 1.0,
$$

with spacing

$$
\Delta\theta
=
\frac{1.0-0.01}{M-1}.
$$

### Step 2 — Prior Initialization

Evaluate the initial Beta prior at every grid point:

```python
prior = stats.beta.pdf(theta_grid, 8, 1.5)
```

Normalize the prior using the trapezoidal rule so that the total probability equals one.

### Step 3 — Sequential Bayesian Update

For each new sensor measurement **yₖ**:

1. Evaluate the log-normal likelihood at every grid point:

```python
likelihood = stats.lognorm.pdf(
    y_k,
    s=sigma,
    scale=theta_grid * K_nominal
)
```

2. Compute the unnormalized posterior:

```python
posterior_unnorm = prior * likelihood
```

3. Normalize the posterior using the trapezoidal rule:

```python
Z = np.trapezoid(posterior_unnorm, theta_grid)
posterior = posterior_unnorm / Z
```

4. Use the normalized posterior as the prior for the next observation:

```python
prior = posterior
```

### Step 4 — Boundary Handling

The parameter θ is physically restricted to the interval

$$
0.01 \le \theta \le 1.0.
$$

Using **0.01** instead of **0** avoids the logarithmic singularity when evaluating the log-normal likelihood. Probability density outside this interval is assumed to be zero and is not included in the numerical integration.

### Step 5 — Running Point Estimates

After each normalization step, compute the running estimators:

```python
theta_bayes = np.trapezoid(theta_grid * posterior, theta_grid)
theta_map = theta_grid[np.argmax(posterior)]
```

The posterior mean provides the Bayesian point estimate, while the MAP estimate identifies the most probable remaining stiffness efficiency.

## Task 6 — Performance Tracking and Degradation Convergence Analysis

In [6]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# ============================================================
# PARAMETERS
# ============================================================

np.random.seed(7)

theta_true = 0.68
K_nominal = 50.0      # kN/mm
sigma = 0.15
n = 15

milestones = {0, 1, 2, 5, 10, 15}

# ============================================================
# COMPUTATIONAL GRID
# ============================================================

theta_grid = np.linspace(0.01, 1.0, 500)

# ============================================================
# INITIAL PRIOR
# ============================================================

prior = stats.beta.pdf(theta_grid, 8, 1.5)
prior /= np.trapezoid(prior, theta_grid)

posterior = prior.copy()

# Store posterior curves
curve_records = {0: posterior.copy()}

# Running estimators
bayes_track = [
    np.trapezoid(theta_grid * posterior, theta_grid)
]

map_track = [
    theta_grid[np.argmax(posterior)]
]

# ============================================================
# SEQUENTIAL BAYESIAN UPDATING
# ============================================================

for k in range(1, n + 1):

    # Generate noisy structural measurement
    noise = np.random.normal(0, sigma)
    y_k = theta_true * K_nominal * np.exp(noise)

    # Expected stiffness for each theta
    expected_K = theta_grid * K_nominal

    # Log-normal likelihood
    likelihood = stats.lognorm.pdf(
        y_k,
        s=sigma,
        scale=expected_K
    )

    # Bayesian update
    posterior *= likelihood

    # Numerical normalization
    posterior /= np.trapezoid(
        posterior,
        theta_grid
    )

    # Running posterior mean
    theta_bayes = np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )

    # Running MAP
    theta_map = theta_grid[
        np.argmax(posterior)
    ]

    bayes_track.append(theta_bayes)
    map_track.append(theta_map)

    # Save milestone distributions
    if k in milestones:
        curve_records[k] = posterior.copy()

# ============================================================
# RESULTS
# ============================================================

print("Running Posterior Mean:")
print(np.round(bayes_track, 3))

print()

print("Running MAP Estimate:")
print(np.round(map_track, 3))

Running Posterior Mean:
[0.842 0.875 0.795 0.761 0.752 0.721 0.714 0.709 0.683 0.694 0.699 0.691
 0.689 0.692 0.689 0.687]

Running MAP Estimate:
[0.933 0.913 0.79  0.754 0.746 0.716 0.71  0.706 0.681 0.692 0.696 0.689
 0.687 0.691 0.689 0.687]


In [7]:
# --- Plot 1: posterior density evolution at milestones ---
fig1 = go.Figure()
for k, curve in curve_records.items():
    fig1.add_trace(go.Scatter(x=theta_grid, y=curve, mode='lines', name=f"Step {k}"))

fig1.add_vline(
    x=theta_true, line_dash="dot", line_color="red",
    annotation_text=f"True θ = {theta_true}", annotation_position="top left"
)
fig1.update_layout(
    title="Posterior Density Progression Across Sensor Readings",
    xaxis_title="Remaining Stiffness Efficiency (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    hovermode="x unified"
)
fig1.show()

In [8]:
# --- Plot 2: convergence timeline ---
steps = list(range(n + 1))

fig2 = go.Figure()
fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="red",
    annotation_text=f"True θ = {theta_true}", annotation_position="bottom right"
)
fig2.add_trace(go.Scatter(x=steps, y=bayes_track, mode='lines+markers', name="Posterior Mean (θ̂_Bayes)"))
fig2.add_trace(go.Scatter(x=steps, y=map_track, mode='lines+markers', name="MAP Estimate (θ̂_MAP)"))

fig2.update_layout(
    title="Convergence of Latent Stiffness Estimators Over Inspections",
    xaxis_title="Sensor Reading Step (k)",
    yaxis_title="Estimated Remaining Stiffness (θ̂)",
    template="plotly_white",
    hovermode="x unified"
)
fig2.show()

### Analysis

At **k = 0**, the prior mean (approximately **0.84**) is well above the true remaining stiffness efficiency (**θ = 0.68**), reflecting the optimistic "healthy" assumption represented by the **Beta(8, 1.5)** prior.

Because each log-normal likelihood is relatively concentrated (**σ = 0.15** in log-space), a sequence of consistently lower sensor measurements quickly shifts the posterior distribution toward the true structural condition. Typically, after about **3–5** sensor readings, both the **Posterior Mean** and the **MAP estimate** move close to **0.68**, indicating that the accumulated sensor data has largely overcome the influence of the initial optimistic prior.

By **k = 15**, both the **Posterior Mean** and the **MAP estimate** fluctuate only slightly around **0.68**, while the posterior density becomes noticeably narrower with each successive update. The taller peak and smaller spread indicate that uncertainty is steadily decreasing as additional measurements are incorporated.

This reduction in uncertainty is an important indicator for structural health monitoring. It shows not only that the estimated remaining stiffness is approaching the true damage level, but also that confidence in the estimate is increasing. As a result, engineers can move from simply suspecting structural damage to confidently identifying that the component has reached approximately **68%** of its original stiffness, providing reliable evidence for maintenance or safety decisions.